# EDA - Clasificacion de letalidad 2026

Este notebook resume el cambio metodologico del proyecto: el objetivo principal es un clasificador binario para estimar `fatalities > 0` usando solo eventos de 2026.

In [ ]:
from pathlib import Path

import pandas as pd

BASE_DIR = Path('..').resolve()
DATA_PATH = BASE_DIR / 'data' / 'processed' / 'model3_embeddings_dataset.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
df['event_date'] = pd.to_datetime(df['event_date'], errors='coerce')
df['year_calc'] = df['event_date'].dt.year
df['fatality_positive'] = (pd.to_numeric(df['fatalities'], errors='coerce').fillna(0) > 0).astype(int)

df.shape

## Distribucion temporal

Comparamos el historico con 2026 para justificar por que el modelo principal se entrena sobre el regimen operativo actual.

In [ ]:
period_summary = (
    df.assign(period=df['year_calc'].where(df['year_calc'].eq(2026), '2024-2025'))
    .groupby('period')
    .agg(
        rows=('fatality_positive', 'size'),
        lethal=('fatality_positive', 'sum'),
        lethal_rate=('fatality_positive', 'mean'),
        fatalities=('fatalities', 'sum'),
    )
)
period_summary['zeros'] = period_summary['rows'] - period_summary['lethal']
period_summary[['rows', 'zeros', 'lethal', 'lethal_rate', 'fatalities']]

## Foco 2026

El dataset de 2026 tiene menos filas que el historico, pero contiene una proporcion letal mucho mayor y fuentes mas alineadas con el problema actual.

In [ ]:
df_2026 = df[df['year_calc'].eq(2026)].copy()

source_summary = (
    df_2026.groupby('source')
    .agg(
        rows=('fatality_positive', 'size'),
        lethal=('fatality_positive', 'sum'),
        lethal_rate=('fatality_positive', 'mean'),
        fatalities=('fatalities', 'sum'),
        max_fatalities=('fatalities', 'max'),
    )
    .sort_values('rows', ascending=False)
)
source_summary

In [ ]:
monthly_summary = (
    df_2026.groupby(df_2026['event_date'].dt.to_period('M'))
    .agg(
        rows=('fatality_positive', 'size'),
        lethal=('fatality_positive', 'sum'),
        lethal_rate=('fatality_positive', 'mean'),
        fatalities=('fatalities', 'sum'),
        max_fatalities=('fatalities', 'max'),
    )
)
monthly_summary

## Variables explicativas clave

In [ ]:
def categorical_lethality(column, top=12):
    summary = (
        df_2026.groupby(column)
        .agg(rows=('fatality_positive', 'size'), lethal=('fatality_positive', 'sum'), lethal_rate=('fatality_positive', 'mean'))
        .sort_values('rows', ascending=False)
        .head(top)
    )
    return summary

categorical_lethality('weapon_type')

In [ ]:
categorical_lethality('target_type')

## Lectura para modelado

- La validacion principal debe ser temporal: entrenar hasta abril y probar en mayo.
- El split aleatorio estratificado se conserva solo como diagnostico.
- La metrica central no debe ser accuracy cruda, sino ROC-AUC, Average Precision, Balanced Accuracy, Precision, Recall y F1.
- El umbral 0.5 es inicial; el siguiente paso natural es calibrarlo segun el costo de falsas alarmas y falsos negativos.